# 7.2 Skills 编排原理

## 概述

上一节我们认识了工作流的五个阶段以及它们之间的关系。

这套工作流的基本编排方式是：

1. `AGENTS.md` 规定阶段顺序：设计 → 生成 → ST → 调优 → 代码 Review。
2. 各阶段 Skill 提供专业方法。
3. Agent 读取输入、形成产物，再根据结果继续或返回。

本节沿用这套流程骨架，继续学习每个阶段是怎样完成工作的，重点学习设计、ST、调优阶段的核心概念。

### 学习目标

- 理解算子设计的基本方法。
- 理解 ST 的覆盖设计和测试可信度。
- 理解算子调优中的 ledger 与门禁约束。

[本章导航](README.md) · [上一节](07.01_overview.ipynb) · [下一节](07.03_practice.ipynb)

本节约 **30～35 分钟**

## 1. 设计阶段：规定算子行为

设计阶段先确认算子“要做什么”，再决定“准备怎样实现”。它关心的是算子信息是否完整、明确，不限定这些信息必须采用哪一种输入形式。

**阶段输入**

设计所需的核心信息包括：

- 算子名称和计算语义。
- 输入、输出的 shape、dtype 和动态维度。
- 编程模式偏好。
- 可选的典型配置、参考实现和性能目标。

这些信息可以直接写在 prompt 中，也可以来自需求文档、算子原型、参考实现、调用位置或已有测试。无论采用哪种形式，开始设计前都要对算子的外部行为形成明确共识。

**主要工作**

- 明确对外行为：
    - 算子功能
    - 输入与输出
    - 精度要求
    - 异常处理
    - 状态修改等副作用
- 做出内部实现选择：
    - 任务如何划分
    - 数据如何搬运
    - 采用什么计算顺序和中间精度
    - 结果如何写回
    - 边界和尾块如何处理
- 设计验证方案。

**如果输入信息不完整，Agent 会从工程中收集信息，并按以下顺序补齐依据：**

1. **收集信息**：查看算子原型、框架文档、PyTorch、CPU 或 CUDA 参考实现、调用位置和已有测试。
2. **交叉核对**：确认不同来源是否描述了相同的输入、计算和输出行为。
3. **识别缺项**：检查算子名称、公式、输入输出规格和编程模式等必需信息是否齐全。
4. **必要时询问**：如果某项不确定性会改变算子结果或支持范围，应向用户确认。

**以 Reduce 为例**

- 如果归约轴能从接口和真实调用中唯一确定，Agent 可以据此补全需求信息。
- 如果不同调用使用不同归约轴，而任务没有限定范围，就需要向用户确认这项外部行为约定。
- tile 大小、每个任务处理多少行、如何搬运、如何计算等属于内部选择，可以由 Agent 结合硬件能力设计。

需求信息明确后，再把“算什么”落实为“怎样计算”，如下图：

![需求中的计算要求怎样落实为分块与数据流设计](images/07.02_design.png)

**检查 `design.md` 时，沿着一份数据追问：**

- 由哪个任务读取？
- 搬到哪里计算？
- 采用什么计算顺序和精度？
- 结果写回哪里？
- 尾块怎样处理？

*`tilelang-op-design` 还会结合 `npu-arch` 核对目标硬件、任务数量、局部空间和可用后端能力。*

**阶段输出**

- `design.md`：记录已经确认的算子行为、实现方案和验证方案。

当前格式可参考课程仓库中的设计模板（`ops/tilelang-op-design/Ascend950/assets/design-template.md`）。


## 2. 生成：把设计落实为可运行实现

**阶段输入**

- `design.md`。
- 仓库中的接口、代码和测试规范。

**主要工作**

- 按设计编写 kernel。
- 编写或接入指定案例的 pytest。
- 完成编译、运行和初步调试。

生成阶段按照下面的过程推进：

![生成阶段从设计到指定案例通过](images/07.02_generation_flow.svg)

**阶段输出**

- 可运行的算子代码。
- 指定案例的 pytest。
- 对应的执行结果。

**进入下一阶段的条件**

指定案例已经实际执行并通过。这个结果只能说明实现具备基本可运行性，测试是否充分还要交给 ST 检查。

## 3. ST：设计完整、可信的算子测试

ST（System Test，系统测试）负责用系统、可复现的测试验证算子是否满足需求。生成阶段的指定案例通过，只能说明实现具备基本可运行性；ST 还要检查算子承诺的行为是否得到充分验证。

这个阶段有两个核心问题：

- **覆盖设计**：应该测试哪些行为，避免遗漏功能、边界或执行路径。
- **测试可信度**：已有测试究竟能够证明什么，避免错误实现因为断言过弱而通过。

覆盖不完整会留下未经验证的区域；测试不可信则可能得到错误的通过结论。因此，ST 既要补齐适用的测试，也要检查每条测试是否真的有效。

**阶段输入**

- 已确认的算子要求与 `design.md`。
- 当前算子代码。
- 已有 pytest 及其执行结果。

**主要工作**

- 判断哪些覆盖点适用于当前算子。
- 检查现有测试能否发现目标错误。
- 补充缺失的测试并实际执行。
- 区分实现错误、编译问题、设备问题和测试未执行。

**覆盖设计**

覆盖设计是指：把算子的要求和风险设计成具体的用例，如下图：

![从算子要求形成有效测试](images/07.02_st_coverage_flow.png)

`tilelang-op-test-design` 会从三组问题共十个点检查是否有遗漏：

- **结果是否正确**：功能、精度、梯度。
- **输入与边界是否正确处理**：边界、异常拒绝、状态修改、布局与接口。
- **实际运行是否符合预期**：后端与分支、随机性、执行证据。

*这些覆盖点不是每个算子都必须机械补齐。Agent 要判断它们是否适用，并为适用项说清“验证什么、为什么这样测、怎样才算通过”。*

**测试可信度**

一条测试要具备可信度，需要同时满足以下四点：

1. **预期有明确依据**：所有预期都应来自公式、算子原型、明确的用户约定或经过核对的参考实现。
2. **参考独立**：不能复制被测 kernel 的同一套错误逻辑进行无效对比。
3. **断言有效**：断言确实能够发现这条用例针对的错误。
4. **真实执行**：测试结果确实是当前代码在目标设备跑出来的结果。

**以行 Softmax 为例，可以按以下关系检查 ST 测试是否可信：**

- 要求：同一行输入相等时，输出应为均匀分布。
- 输入：构造一行相同的数值。
- 参考：用独立的 PyTorch 实现计算完整结果。
- 断言：按规定容差比较每个输出元素。
- 证据：保存实际执行的 pytest 节点与结果。

**阶段输出**

- 可以持续运行的补充测试。
- 覆盖结论。
- 与当前代码对应的执行证据。

**需要返回时**

- 发现实现错误：返回生成阶段修复。
- 发现需求依据冲突：返回设计阶段确认。

完整方法可查看课程仓库中的覆盖设计（`ops/tilelang-op-test-design/Ascend950/references/coverage-design.md`）与测试可信度（`ops/tilelang-op-test-design/Ascend950/references/test-credibility.md`）。

## 4. 调优：提升算子性能

**阶段输入**

- 已经通过 ST 的实现。
- 完整的性能案例与目标。
- 源码、profiling 数据和设备信息。

**主要工作**

- 固定一份可回退的性能基线。
- 根据性能事实提出优化方向。
- 建立、实现并验证候选。
- 根据正确性和性能结果采用或淘汰候选。

**优化候选从哪里来**

候选不能只来自猜测。Agent 先用性能测试工具采集数据，重点收集两类事实：

- **源码事实**：任务怎样划分、数据怎样搬运、是否存在重复计算。
- **实测事实**：时间花在哪里，哪些 case 慢，瓶颈随 shape 怎样变化。

两类事实能够相互印证后，再形成待验证方向。一次优化按以下顺序推进：

![从性能事实形成并筛选优化候选](images/07.02_tuning_candidate_flow.png)

**用 ledger 和门禁防止遗漏**

调优过程中，Agent 先把 case、待验证方向、候选和结果持续记入一份 ledger，主要记录四类内容：

- **case**：每个性能案例的结构与实测性能数据。
- **待验证方向**：根据这些数据给出可能的优化方向。
- **候选**：准备修改什么、适用于哪些 case，记为 `CREATED`。
- **结果**：候选的编译、正确性和性能结果，记为 `RESULT`。

结果必须绑定到具体候选和具体 case。一个候选失败，只能否定这次**候选**，不能直接否定**方向**。

门禁脚本在三个关键位置检查 ledger：

1. **第一次修改前**：检查目标 case 和待验证方向是否已经记录，避免只实现最先想到的方案。
2. **创建下一个候选前**：检查候选身份、适用 case 和新增记录是否前后一致，避免方案与结论不一致。
3. **最终停止前**：执行最终检查；仍有方向未裁决、候选缺少结果或记录没有闭环时，不能宣布调优完成。

ledger 负责保存过程事实，门禁负责检查 ledger 是否完整、一致。

**阶段输出**

- 通过正确性与性能门禁的最终代码。
- 完整案例的性能对比。
- 候选搜索记录与性能报告。

**需要返回时**

- 正确性失败：修复或淘汰当前候选。
- 性能未达目标：保留可靠基线，继续分析候选。

Flash 与 Standard 的组织方式不同，但都不能跳过基线、完整案例、正确性回归和同口径性能比较。方法原文见课程仓库中的 Flash 工作流（`plugins-official/tilelang-op-orchestrator/workflows/Ascend950/workflows/tilelang-tuning-flash.md`）和性能优化搜索覆盖门禁（`plugins-official/tilelang-op-orchestrator/workflows/Ascend950/workflows/optimization-search-coverage/optimization_search_coverage_gate.md`）。

## 5. 代码 Review 与最终交付

**阶段输入**

- 最终 kernel、pytest 和辅助代码。
- ST 与性能结果。
- 仓库的格式、lint 与 CI 规范。

**主要工作**

- 检查代码是否符合仓库规范。
- 确认测试和性能证据对应最终代码。
- 汇总交付结论与适用范围。

**阶段输出**

- 格式检查报告。
- 最终代码及测试。
- ST、性能与交付结论。

## 小结

- 设计要敲定外部行为并选择合理的内部实现。
- ST 通过覆盖设计和测试可信度，提高算子质量验证的完整性和可信度。
- 调优提出候选，并用 ledger 与门禁约束搜索过程，减少遗漏和前后矛盾。
- 代码 Review 检查最终代码及证据，完成交付收口。

## 课后练习

1. 某行包含 N 个相同的输入值。该行的 Softmax 预期输出是多少？测试应该怎样比较实际输出与预期输出？可以只检查输出的 shape 吗？
2. Agent 创建了两个优化候选，但 ledger 中只有一个候选的 `RESULT`。最终门禁能帮助发现什么？

运行下面的代码单元即可查看练习解析。


In [ ]:
from pathlib import Path
print(Path("answer/07.02_answer.md").read_text(encoding="utf-8"))


[上一节：看懂 Agent 整体架构](07.01_overview.ipynb) · [返回章节目录](README.md) · [下一节：一个 Softmax 算子的交付过程](07.03_practice.ipynb)